# Libs and codes

In [ ]:
#!pip install python-docx

In [36]:
"""
Script simplificado para generar documentos Word
con información de avisos de ceniza volcánica del VAAC
"""

import requests
from bs4 import BeautifulSoup
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
import re
import os

class VAACDocumentGenerator:
    def __init__(self):
        self.base_url = "https://www.ospo.noaa.gov"
        self.main_page = f"{self.base_url}/products/atmosphere/vaac/messages.html"
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })

    # def construir_url_advisory(self, codigo_volcan, fecha, hora_utc):
    #     """
    #     Construye la URL del advisory a partir de los datos del volcán
    #     Ejemplo: PURA, '29 Nov 2025', '0803 UTC' ->
    #     https://www.ospo.noaa.gov/VAAC/ARCH25/PURA/2025K290803.html
    #     """
    #     # Convertir fecha a formato de URL
    #     # '29 Nov 2025' -> '290803' (día + hora sin espacios)
    #     dia = fecha.split()[0].zfill(2)  # Asegurar 2 dígitos
    #     hora = hora_utc.replace(' UTC', '').replace(':', '')  # Remover UTC y :

    #     # Obtener año de la fecha
    #     partes_fecha = fecha.split()
    #     ano = partes_fecha[-1]  # 2025

    #     # Construir nombre de archivo: 2025K290803.html
    #     nombre_archivo = f"{ano}L{dia}{hora}.html"

    #     # Mapeo de códigos de volcán a nombres de carpeta
    #     mapeo_carpetas = {
    #         'FUEG': 'FUEG',
    #         'PURA': 'PURA',
    #         'REVE': 'REVE',
    #         'RUIZ': 'RUIZ',
    #         'SANT': 'SANTAMA',
    #         'SANGAY3': 'SANGAY3'
    #     }

    #     carpeta = mapeo_carpetas.get(codigo_volcan, codigo_volcan)

    #     url = f"{self.base_url}/VAAC/ARCH25/{carpeta}/{nombre_archivo}"
    #     return url


    def construir_url_advisory(self, codigo_volcan, fecha, hora_utc):
        """
        Construye la URL del advisory a partir de los datos del volcán
        Ejemplo: PURA, '29 Nov 2025', '0803 UTC' ->
        https://www.ospo.noaa.gov/VAAC/ARCH25/PURA/2025K290803.html
        """
        # Convertir fecha a formato de URL
        # '29 Nov 2025' -> '290803' (día + hora sin espacios)
        dia = fecha.split()[0].zfill(2)  # Asegurar 2 dígitos
        hora = hora_utc.replace(' UTC', '').replace(':', '')  # Remover UTC y :

        # Obtener año y mes de la fecha
        partes_fecha = fecha.split()
        ano = partes_fecha[-1]  # 2025
        mes_str = partes_fecha[1]  # 'Nov', 'Dec', etc.

        # Mapeo de mes a letra (A=Enero, B=Febrero, ..., L=Diciembre)
        meses_a_letra = {
            'Jan': 'A', 'Feb': 'B', 'Mar': 'C', 'Apr': 'D',
            'May': 'E', 'Jun': 'F', 'Jul': 'G', 'Aug': 'H',
            'Sep': 'I', 'Oct': 'J', 'Nov': 'K', 'Dec': 'L'
        }

        letra_mes = meses_a_letra.get(mes_str, 'K')  # Por defecto K si no se encuentra

        # Construir nombre de archivo: 2025K290803.html (K para noviembre)
        nombre_archivo = f"{ano}{letra_mes}{dia}{hora}.html"

        # Mapeo de códigos de volcán a nombres de carpeta
        mapeo_carpetas = {
            'FUEG': 'FUEG',
            'PURA': 'PURA',
            'REVE': 'REVE',
            'RUIZ': 'RUIZ',
            'SANT': 'SANTAMA',
            'SANGAY3': 'SANGAY3'
        }

        carpeta = mapeo_carpetas.get(codigo_volcan, codigo_volcan)

        url = f"{self.base_url}/VAAC/ARCH{ano[-2:]}/{carpeta}/{nombre_archivo}"
        return url


    def extraer_numero_advisory(self, texto_advisory):   
        """
        Extrae el número del advisory del texto
        Ejemplo: "ADVISORY NR: 2025/192" -> "0192"
        """
        match = re.search(r'ADVISORY NR:\s*(\d+)/(\d+)', texto_advisory)
        if match:
            numero = match.group(2)
            return numero.zfill(4)       ### cambiar a 4 cuando corresponda
        return None

    def construir_url_imagen(self, codigo_volcan, numero_advisory):
        """
        Construye la URL de la imagen usando el código del volcán y número de advisory
        Ejemplo: PURA + 0192 -> https://www.ospo.noaa.gov/VAAC/ARCH25/GFX/PURA0192.jpg
        """
        if codigo_volcan and numero_advisory:
            # Casos especiales
            if codigo_volcan == 'SANTAMA':
                codigo_volcan = 'SANT'
            elif codigo_volcan == 'SANGAY3':
                codigo_volcan = 'SANG'

            return f"{self.base_url}/VAAC/ARCH26/GFX/{codigo_volcan}{numero_advisory}.jpg"
            #{self.ano[-2:]}
        return None

    def limpiar_texto_advisory(self, texto):
        """
        Limpia el texto del advisory eliminando elementos no deseados
        """
        if "Full Size Graphic" in texto:
            texto = texto[:texto.find("Full Size Graphic")].strip()

        lineas = texto.split('\n')
        lineas_limpias = []

        for linea in lineas:
            linea_limpia = linea.strip()
            if linea_limpia and linea_limpia != "-":
                lineas_limpias.append(linea)

        return '\n'.join(lineas_limpias).strip()


    def unir_lineas_coordenadas(self, lineas):
        nuevas = []
        i = 0
    
        while i < len(lineas):
            actual = lineas[i].strip()
    
            if i < len(lineas) - 1:
                siguiente = lineas[i + 1].strip()
    
                # 👉 Caso clave: S0007 + W07741...
                if re.match(r'^[NS]\d{2,4}$', actual) and re.match(r'^[EW]\d{3,5}', siguiente):
                    nuevas.append(actual + " " + siguiente)
                    i += 2
                    continue
    
            nuevas.append(actual)
            i += 1
    
        return nuevas

        
    def extraer_datos_advisory(self, url, codigo_volcan):
        """
        Extrae el texto del advisory y construye la URL de la imagen
        """
        try:
            response = self.session.get(url)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            # Extraer el texto del advisory
            texto_advisory = ""
            pre_tag = soup.find('pre')
            if pre_tag:
                texto_advisory = pre_tag.text.strip()

            if not texto_advisory:
                body_text = soup.get_text()
                if "VA ADVISORY" in body_text:
                    start_idx = body_text.find("FVXX")
                    if start_idx != -1:
                        end_idx = body_text.find("Full Size Graphic")
                        if end_idx == -1:
                            end_idx = len(body_text)
                        texto_advisory = body_text[start_idx:end_idx].strip()

            # Limpiar el texto
            texto_advisory = self.limpiar_texto_advisory(texto_advisory)

            # Extraer número de advisory y construir URL de imagen
            numero_advisory = self.extraer_numero_advisory(texto_advisory)
            url_imagen = self.construir_url_imagen(codigo_volcan, numero_advisory)

            return {
                'texto': texto_advisory,
                'url_imagen': url_imagen,
                'url_fuente': url
            }

        except Exception as e:
            print(f"Error al extraer datos del advisory: {e}")
            return None

    def descargar_imagen(self, url_imagen, nombre_archivo):
        """
        Descarga una imagen desde una URL
        """
        try:
            response = self.session.get(url_imagen, stream=True)
            response.raise_for_status()

            with open(nombre_archivo, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)

            return nombre_archivo

        except Exception as e:
            print(f"Error al descargar imagen: {e}")
            return None

    def crear_documento_word(self, volcan_info, advisory_data, nombre_archivo):
        """
        Crea un documento Word SOLO con el texto del advisory y la imagen
        """
        try:
            from docx.shared import RGBColor
            from docx.enum.text import WD_LINE_SPACING

            doc = Document()

            # Procesar el texto línea por línea para aplicar formato especial
            lineas = advisory_data['texto'].split('\n')
            # 👇 ESTA ES LA LÍNEA NUEVA (EL FIX REAL)
            lineas = self.unir_lineas_coordenadas(lineas)
            palabras_clave = ['VOLCANO:', 'ADVISORY NR:', 'NXT ADVISORY:']

            # Crear un solo párrafo para todo el texto
            parrafo = doc.add_paragraph()
            parrafo.paragraph_format.line_spacing_rule = WD_LINE_SPACING.SINGLE
            parrafo.paragraph_format.space_before = Pt(0)
            parrafo.paragraph_format.space_after = Pt(0)

            en_bloque_cld = False  # <-- añadir ANTES del for
            
            for i, linea in enumerate(lineas):
                linea_stripped = linea.strip()
                # Verificar si la línea contiene alguna palabra clave
                contiene_palabra_clave = any(palabra in linea for palabra in palabras_clave)

                if contiene_palabra_clave:
                    # Aplicar formato especial: negrita y resaltado amarillo
                    run = parrafo.add_run(linea)
                    run.font.name = 'Courier New'
                    run.font.size = Pt(9)
                    run.font.bold = True
                    run.font.highlight_color = 7  # 7 = Amarillo (WD_COLOR_INDEX.YELLOW)
                else:
                    # Formato normal
                    run = parrafo.add_run(linea)
                    run.font.name = 'Courier New'
                    run.font.size = Pt(9)

                # =========================
                # LÓGICA INTELIGENTE DE UNIÓN
                # =========================
                if i < len(lineas) - 1:
                    siguiente = lineas[i + 1].strip()
                    unir = False

                    # # 🚫 NO unir casos explícitos
                    # if "NO VA EXP" in linea_stripped:
                    #     unir = False
                
                    # # 🚨 RMK siempre nueva línea
                    # elif siguiente.startswith("RMK"):
                    #     unir = False
                    
                    # # 1. VA CLD siempre une con lo siguiente
                    # elif linea_stripped.startswith(("OBS VA CLD", "EST VA CLD", "FCST VA CLD")):
                    #     unir = True

                    # # 2. Si la línea actual termina en MOV → unir
                    # elif linea_stripped.endswith("MOV"):
                    #     unir = True

                    # # 3. Si la siguiente línea es coordenada o continuación
                    # # elif re.match(r'^[SWNE]\d', siguiente) or siguiente.startswith("-"):
                    # # elif re.match(r'^[NS]\d{2,4}\s*[EW]\d{3,5}', siguiente) or siguiente.startswith("-"):
                    #     # unir = True
                    # elif (
                    #     # Coordenada completa en una línea
                    #     re.match(r'^[NS]\d{2,4}\s*[EW]\d{3,5}', siguiente)
                    
                    #     # Coordenada partida en 2 líneas (S... + W...)
                    #     or (
                    #         # re.match(r'^[NS]\d{2,4}$', linea_stripped) and
                    #         # re.match(r'^[EW]\d{3,5}', siguiente)
                    #         re.match(r'^-?\s*[NS]\d{2,4}$', linea_stripped) and
                    #         re.match(r'^[EW]\d{3,5}', siguiente)
                    #     )
                    
                    #     # Continuación con guion
                    #     or siguiente.startswith("-")
                    # ):
                    #     unir = True

                    # # 4. (extra útil) líneas que empiezan con AND / TO
                    # elif siguiente.startswith(("AND", "TO")):
                    #     unir = True

                    # if unir:
                    #     parrafo.add_run(" ")
                    # else:
                    #     parrafo.add_run("\n")

                    # Activar modo bloque CLD
                    if linea_stripped.startswith(("OBS VA CLD", "EST VA CLD", "FCST VA CLD")):
                        en_bloque_cld = True
            
                    # Desactivar bloque CLD si la siguiente línea es una sección nueva
                    if en_bloque_cld:
                        es_nueva_seccion = (
                            siguiente.startswith(("OBS VA CLD", "EST VA CLD", "FCST VA CLD",
                                                  "RMK", "NXT ADVISORY", "VOLCANO",
                                                  "ADVISORY NR", "FVXX", "FCST"))
                            or siguiente == ""
                            or "NO VA EXP" in siguiente
                        )
                        if es_nueva_seccion:
                            en_bloque_cld = False
                            unir = False
                        else:
                            unir = True  # <-- dentro del bloque, todo se une
            
                    # Resto de condiciones originales (para líneas fuera de bloque CLD)
                    elif "NO VA EXP" in linea_stripped:
                        unir = False
                    elif siguiente.startswith("RMK"):
                        unir = False
                    elif linea_stripped.endswith("MOV"):
                        unir = True
                    elif (
                        re.match(r'^[NS]\d{2,4}\s*[EW]\d{3,5}', siguiente)
                        or (re.match(r'^-?\s*[NS]\d{2,4}$', linea_stripped) and re.match(r'^[EW]\d{3,5}', siguiente))
                        or siguiente.startswith("-")
                    ):
                        unir = True
                    elif siguiente.startswith(("AND", "TO")):
                        unir = True
            
                    if unir:
                        parrafo.add_run(" ")
                    else:
                        parrafo.add_run("\n")
                    

            # Imagen del área afectada (sin salto de línea adicional)
            if advisory_data['url_imagen']:
                temp_imagen = f"temp_{volcan_info['nombre'].replace(' ', '_')}.jpg"
                imagen_descargada = self.descargar_imagen(advisory_data['url_imagen'], temp_imagen)

                if imagen_descargada and os.path.exists(imagen_descargada):
                    try:
                        # Agregar imagen ajustada al margen sin formato especial
                        doc.add_picture(imagen_descargada, width=Inches(6))
                        print(f"  ✓ Imagen agregada correctamente")
                    except Exception as img_error:
                        print(f"  ✗ Error al insertar imagen en Word: {img_error}")
                    finally:
                        try:
                            os.remove(temp_imagen)
                        except:
                            pass
                else:
                    print(f"  ✗ No se pudo descargar la imagen desde: {advisory_data['url_imagen']}")

            doc.save(nombre_archivo)
            print(f"✓ Documento creado: {nombre_archivo}")
            return True

        except Exception as e:
            print(f"Error al crear documento Word: {e}")
            return False


    def construir_ruta_salida(self, nombre_volcan_solo, volcan_in_peru, anho, mes, volcanes):

        main_path = "D:/OVM/2-REGISTROS-VA_ADVISORY/"
        #main_path = "D:/PRONOSTICADORES/JMG/ADVISORIES/"
    
        # PERÚ / OTROS
        if volcan_in_peru == "Y":
            peru_ono = "1-PERU"
        else:
            peru_ono = "2-OTROS"
    
        # AÑO
        path_anho = f"AÑO-{anho}"
    
        # MES
        path_mes = f"{mes}_{anho}"
    
        # VOLCÁN
        if nombre_volcan_solo == "SABANCAYA":
            path_volcan = f"1-{nombre_volcan_solo}"
        elif nombre_volcan_solo == "UBINAS":
            path_volcan = f"2-{nombre_volcan_solo}"
        else:
            path_volcan = nombre_volcan_solo
            if nombre_volcan_solo == "SANTA":
                path_volcan = "SANTA MARIA"
    
        # =========================
        # PERÚ
        # =========================
        if volcan_in_peru == "Y":
            fpath_abs = os.path.join(main_path, peru_ono, path_anho, path_volcan, path_mes)
    
        # =========================
        # OTROS PAÍSES
        # =========================
        else:
            path_pais = "SIN_PAIS"
    
            for key in volcanes.keys():
                if nombre_volcan_solo in volcanes.get(key, []):
                    path_pais = key
                    break
    
            fpath_abs = os.path.join(main_path, peru_ono, path_anho, path_pais, path_volcan, path_mes)
    
        return fpath_abs


    # def convertir_a_pdf(self, ruta_docx):
    #     from docx2pdf import convert
        
    #     try:
    #         ruta_pdf = ruta_docx.replace(".docx", ".pdf")
    #         nruta_pdf = "//PC_GOA76/solicitud_carpeta"+"/"+ruta_pdf.split("/")[-1]
    #         convert(ruta_docx, nruta_pdf)
    #         print(f"✓ PDF generado: {nruta_pdf}")
    #         return ruta_pdf
    #     except Exception as e:
    #         print(f"✗ Error al convertir: {e}")
    #         return None

    def convertir_a_pdf(self, ruta_docx):
        from docx2pdf import convert
        
        try:
            ruta_pdf = ruta_docx.replace(".docx", ".pdf")
            nombre_pdf = os.path.basename(ruta_pdf)  # extrae solo el nombre del archivo
            nruta_pdf = os.path.join(r"\\PC_GOA76\solicitud_carpeta", nombre_pdf)
            convert(ruta_docx, nruta_pdf)
            print(f"✓ PDF generado: {nruta_pdf}")
            return ruta_pdf
        except Exception as e:
            print(f"✗ Error al convertir: {e}")
            return None
    

    def procesar_volcan(self, nombre, codigo_volcan, fecha, hora_utc, carpeta_salida="."):
        """
        Procesa un volcán específico y genera su documento

        Parámetros:
        - nombre: Nombre completo del volcán (ej: 'PURACE COLOMBIA')
        - codigo_volcan: Código de 4 letras del volcán (ej: 'PURA')
        - fecha: Fecha en formato 'DD Mon YYYY' (ej: '29 Nov 2025')
        - hora_utc: Hora en formato 'HHMM UTC' (ej: '0803 UTC')
        """
        print(f"\nProcesando: {nombre} - {hora_utc}")

        # Construir URL del advisory
        url = self.construir_url_advisory(codigo_volcan, fecha, hora_utc)
        print(f"URL generada: {url}")

        # Extraer datos del advisory
        advisory_data = self.extraer_datos_advisory(url, codigo_volcan)

        if not advisory_data or not advisory_data['texto']:
            print(f"✗ No se pudo extraer información del advisory")
            return False

        # Crear carpeta de salida
        #os.makedirs(carpeta_salida, exist_ok=True)

        # Extraer número de advisory del texto
        numero_advisory = self.extraer_numero_advisory(advisory_data['texto'])
        if not numero_advisory:
            numero_advisory = "0000"  # Valor por defecto si no se encuentra

        # Extraer solo el nombre del volcán (sin el país)
        # Ejemplo: 'PURACE COLOMBIA' -> 'PURACE'
        nombre_volcan_solo = nombre.split()[0]

        # Generar nombre de archivo: VOLCANNAME_ADV_NUMADV.docx
        # Ejemplo: PURACE_ADV_0192.docx
        #nombre_archivo = f"{carpeta_salida}/{nombre_volcan_solo}_ADV_{numero_advisory}.docx"

        # ===============================================================================================
        # ===== CONFIG =====
        volcan_in_peru = "N"  # <-- puedes automatizar luego
        anho = int(fecha.split()[-1])  # 2026
        mes_map = {
            "Jan": "1-ENE", "Feb": "2-FEB", "Mar": "3-MAR", "Apr": "4-ABR",
            "May": "5-MAY", "Jun": "6-JUN", "Jul": "7-JUL", "Aug": "8-AGO",
            "Sep": "9-SEP", "Oct": "10-OCT", "Nov": "11-NOV", "Dec": "12-DIC"
        }
        mes_str = fecha.split()[1]
        mes = mes_map.get(mes_str, mes_str)
        
        # Diccionario volcanes (ejemplo)
        volcanes = {
            "ECUADOR": ["COTOPAXI","REVENTADOR","SANGAY"],
            "COLOMBIA": ["RUIZ", "PURACE"],
            "GUATEMALA": ["FUEGO", "SANTA", "PACAYA"],
            "MEXICO": ["POPOCATEPETL"]
        }
        
        # ===== RUTA FINAL =====
        fpath_abs = self.construir_ruta_salida(
            nombre_volcan_solo,
            volcan_in_peru,
            anho,
            mes,
            volcanes
        )
        
        # Crear carpetas
        os.makedirs(fpath_abs, exist_ok=True)
        
        # Nombre final
        nombre_archivo = os.path.join(
            fpath_abs,
            f"{nombre_volcan_solo}_ADV_{numero_advisory}.docx"
        )

        # ===============================================================================================

        if os.path.exists(nombre_archivo):
            print(f"⚠ Ya existe: {nombre_archivo}, omitiendo...")
            return False
        
        # Crear documento
        volcan_info = {
            'nombre': nombre,
            'fecha': fecha,
            'hora_utc': hora_utc,
            'url': url
        }

        #return self.crear_documento_word(volcan_info, advisory_data, nombre_archivo)

        exito = self.crear_documento_word(volcan_info, advisory_data, nombre_archivo)
        if exito:
            self.convertir_a_pdf(nombre_archivo) #.split("/")[-1]
        return exito


    def mostrar_advisories_disponibles(self, limite=10):
        """
        Muestra los advisories disponibles en la página web para copiar y usar
        """
        try:
            print(f"\n{'='*80}")
            print(f"ADVISORIES DISPONIBLES EN VAAC")
            print(f"{'='*80}\n")

            response = self.session.get(self.main_page)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            # Buscar "Advisories Last Updated"
            advisories_updated = soup.find('strong', string=lambda text: text and 'Advisories Last Updated' in text)
            if advisories_updated:
                print(f"📅 {advisories_updated.text.strip()}\n")

            # Buscar el div con clase "bg-medium-gray"
            bg_div = soup.find('div', class_='bg-medium-gray')
            if not bg_div:
                print("No se encontró el contenedor de advisories")
                return

            # Encontrar las dos columnas
            columnas = bg_div.find_all('div', class_='column')
            if len(columnas) < 2:
                print("No se encontraron las columnas de datos")
                return

            columna_volcanes = columnas[0]  # Primera columna: nombres de volcanes
            columna_horas = columnas[1]     # Segunda columna: fechas y horas

            # Extraer enlaces de volcanes
            enlaces_volcanes = columna_volcanes.find_all('a')

            # Extraer información de horas (cada línea tiene fecha + hora + enlaces)
            lineas_hora = []
            for line in columna_horas.find_all('br'):
                # Obtener el contenido antes del <br>
                prev = line.previous_sibling
                if prev and isinstance(prev, str):
                    continue

            # Mejor enfoque: obtener todo el contenido de la columna de horas
            hora_html = str(columna_horas)
            # Dividir por <br> y procesar cada línea
            lineas = hora_html.split('<br>')

            advisories = []

            # Mapeo de nombres de volcanes a códigos
            mapeo_nombres_codigos = {
                'FUEGO GUATEMALA': 'FUEG',
                'PURACE COLOMBIA': 'PURA',
                'REVENTADOR ECUADOR': 'REVE',
                'RUIZ COLOMBIA': 'RUIZ',
                'SANTA MARIA GUATEMALA': 'SANT',
                'SANGAY ECUADOR': 'SANGAY3',
                'KILAUEA HAWAIIAN IS': 'KILAUEA',
                'KILAUEA HAWAIIAN.IS': 'KILAUEA',
                'POPOCATEPETL MEXICO': 'POPO'
            }

            print("Últimos advisories encontrados:\n")

            for idx, (enlace_volcan, linea_hora) in enumerate(zip(enlaces_volcanes, lineas[:len(enlaces_volcanes)])):
                if idx >= limite:
                    break

                nombre_volcan = enlace_volcan.text.strip()

                # Extraer fecha y hora de la línea
                soup_linea = BeautifulSoup(linea_hora, 'html.parser')

                # Buscar <em> para fecha
                em_tag = soup_linea.find('em')
                if not em_tag:
                    continue

                fecha_texto = em_tag.text.strip().replace(' - ', '').replace(' -', '').replace('-', '').strip()

                # Buscar primer <a> para hora UTC
                a_tag = soup_linea.find('a', href=True)
                if not a_tag:
                    continue

                hora_utc = a_tag.text.strip()

                # Buscar el código del volcán
                codigo_volcan = mapeo_nombres_codigos.get(nombre_volcan, '????')

                contador = idx + 1

                # Mostrar información formateada
                print(f"[{contador}] {nombre_volcan}")
                print(f"    Código: {codigo_volcan}")
                print(f"    Fecha:  {fecha_texto}")
                print(f"    Hora:   {hora_utc}")
                print(f"    Copiar: generador.procesar_volcan('{nombre_volcan}', '{codigo_volcan}', '{fecha_texto}', '{hora_utc}')")
                print()

                advisories.append((nombre_volcan, codigo_volcan, fecha_texto, hora_utc))

            print(f"{'='*80}")
            print(f"Total mostrados: {len(advisories)}")
            print(f"{'='*80}\n")

            # Generar código para copiar directamente
            if advisories:
                print("CÓDIGO PARA COPIAR Y PEGAR:\n")
                print("# Procesar volcanes individualmente:")
                for nombre, codigo, fecha, hora in advisories:
                    print(f"generador.procesar_volcan('{nombre}', '{codigo}', '{fecha}', '{hora}')")

                print("\n# O procesar todos en lista:")
                print("ultimos_advisories = [")
                for nombre, codigo, fecha, hora in advisories:
                    print(f"    ('{nombre}', '{codigo}', '{fecha}', '{hora}'),")
                print("]")
                print("generador.procesar_lista_volcanes(ultimos_advisories)")
                print()

        except Exception as e:
            print(f"✗ Error al obtener advisories: {e}")
            import traceback
            traceback.print_exc()

    def obtener_ultimos_advisories(self, limite=5):
        """
        Obtiene los últimos N advisories desde la página web principal
        Retorna una lista de tuplas: (nombre, codigo, fecha, hora_utc)
        """
        try:
            print(f"Obteniendo últimos {limite} advisories desde la web...")
            response = self.session.get(self.main_page)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            advisories = []
            filas = soup.find_all('tr')

            # Mapeo de nombres de volcanes a códigos
            mapeo_nombres_codigos = {
                'FUEGO GUATEMALA': 'FUEG',
                'PURACE COLOMBIA': 'PURA',
                'REVENTADOR ECUADOR': 'REVE',
                'RUIZ COLOMBIA': 'RUIZ',
                'SANTA MARIA GUATEMALA': 'SANT',
                'SANGAY ECUADOR': 'SANGAY3',
                'KILAUEA HAWAIIAN IS': 'KILAUEA',
                'POPOCATEPETL MEXICO': 'POPOCATEPETL'
            }

            for fila in filas:
                if len(advisories) >= limite:
                    break

                celdas = fila.find_all('td')
                if len(celdas) >= 3:
                    # Columna 0: Nombre del volcán
                    nombre_celda = celdas[0].find('a')
                    if nombre_celda:
                        nombre_completo = nombre_celda.text.strip()

                        # Columna 1: Fecha
                        fecha_texto = celdas[1].text.strip()

                        # Columna 2: Hora UTC
                        hora_celda = celdas[2].find('a')
                        if hora_celda:
                            hora_utc = hora_celda.text.strip()

                            # Buscar el código del volcán
                            codigo_volcan = mapeo_nombres_codigos.get(nombre_completo)

                            if codigo_volcan:
                                advisories.append((
                                    nombre_completo,
                                    codigo_volcan,
                                    fecha_texto,
                                    hora_utc
                                ))
                            else:
                                print(f"  ⚠ Volcán no mapeado: {nombre_completo}")

            print(f"✓ Se encontraron {len(advisories)} advisories")
            return advisories

        except Exception as e:
            print(f"✗ Error al obtener advisories: {e}")
            return []

    def procesar_ultimos_advisories(self, limite=5):
        """
        Obtiene y procesa automáticamente los últimos N advisories
        """
        advisories = self.obtener_ultimos_advisories(limite)

        if not advisories:
            print("No se encontraron advisories para procesar")
            return

        print(f"\n=== Procesando {len(advisories)} volcanes ===")

        exitosos = 0
        fallidos = 0

        for i, (nombre, codigo, fecha, hora) in enumerate(advisories, 1):
            print(f"\n[{i}/{len(advisories)}]")
            if self.procesar_volcan(nombre, codigo, fecha, hora):
                exitosos += 1
            else:
                fallidos += 1

        print(f"\n=== Resumen ===")
        print(f"✓ Exitosos: {exitosos}")
        print(f"✗ Fallidos: {fallidos}")

        return exitosos, fallidos

    def procesar_lista_volcanes(self, lista_volcanes):
        """
        Procesa múltiples volcanes desde una lista

        Parámetros:
        - lista_volcanes: Lista de tuplas (nombre, codigo, fecha, hora_utc)

        Ejemplo:
        lista = [
            ('PURACE COLOMBIA', 'PURA', '01 Dec 2025', '1812 UTC'),
            ('REVENTADOR ECUADOR', 'REVE', '01 Dec 2025', '1559 UTC'),
        ]
        """
        print(f"\n=== Procesando {len(lista_volcanes)} volcanes ===")

        exitosos = 0
        fallidos = 0

        for i, (nombre, codigo, fecha, hora) in enumerate(lista_volcanes, 1):
            print(f"\n[{i}/{len(lista_volcanes)}]")
            if self.procesar_volcan(nombre, codigo, fecha, hora):
                exitosos += 1
            else:
                fallidos += 1

        print(f"\n=== Resumen ===")
        print(f"✓ Exitosos: {exitosos}")
        print(f"✗ Fallidos: {fallidos}")

        return exitosos, fallidos


def main():
    """
    Función principal - Ingresa los datos del volcán aquí
    """
    generador = VAACDocumentGenerator()

    # ===== PASO 1: MOSTRAR advisories disponibles en la web =====
    generador.mostrar_advisories_disponibles(limite=5)

    # Después de ver los advisories, descomenta la opción que prefieras:

    # ===== OPCIÓN 1: AUTOMÁTICO - Procesar últimos N advisories desde la web =====
    # generador.procesar_ultimos_advisories(limite=5)

    # ===== OPCIÓN 2: Procesar un solo volcán manualmente =====
    # nombre_volcan = 'PURACE COLOMBIA'
    # codigo_volcan = 'PURA'
    # fecha = '01 Dec 2025'
    # hora_utc = '1812 UTC'
    # generador.procesar_volcan(nombre_volcan, codigo_volcan, fecha, hora_utc)

    # ===== OPCIÓN 3: Procesar lista manual de volcanes (copia desde arriba) =====
    # ultimos_advisories = [
    #     ('PURACE COLOMBIA', 'PURA', '01 Dec 2025', '1812 UTC'),
    #     ('REVENTADOR ECUADOR', 'REVE', '01 Dec 2025', '1559 UTC'),
    # ]
    # generador.procesar_lista_volcanes(ultimos_advisories)


if __name__ == "__main__":
    main()


ADVISORIES DISPONIBLES EN VAAC

📅 Advisories Last Updated:   Sun Apr 19 19:33:53  UTC 2026

Últimos advisories encontrados:

[1] SANTA MARIA GUATEMALA
    Código: SANT
    Fecha:  19 Apr 2026
    Hora:   1930 UTC
    Copiar: generador.procesar_volcan('SANTA MARIA GUATEMALA', 'SANT', '19 Apr 2026', '1930 UTC')

[2] SANGAY ECUADOR
    Código: SANGAY3
    Fecha:  19 Apr 2026
    Hora:   1859 UTC
    Copiar: generador.procesar_volcan('SANGAY ECUADOR', 'SANGAY3', '19 Apr 2026', '1859 UTC')

[3] FUEGO GUATEMALA
    Código: FUEG
    Fecha:  19 Apr 2026
    Hora:   1714 UTC
    Copiar: generador.procesar_volcan('FUEGO GUATEMALA', 'FUEG', '19 Apr 2026', '1714 UTC')

[4] REVENTADOR ECUADOR
    Código: REVE
    Fecha:  19 Apr 2026
    Hora:   1410 UTC
    Copiar: generador.procesar_volcan('REVENTADOR ECUADOR', 'REVE', '19 Apr 2026', '1410 UTC')

[5] SANTA MARIA GUATEMALA
    Código: SANT
    Fecha:  19 Apr 2026
    Hora:   1355 UTC
    Copiar: generador.procesar_volcan('SANTA MARIA GUATEMALA', '

# Ejecutar

In [40]:
main()


ADVISORIES DISPONIBLES EN VAAC

📅 Advisories Last Updated:   Sun Apr 19 19:33:53  UTC 2026

Últimos advisories encontrados:

[1] SANTA MARIA GUATEMALA
    Código: SANT
    Fecha:  19 Apr 2026
    Hora:   1930 UTC
    Copiar: generador.procesar_volcan('SANTA MARIA GUATEMALA', 'SANT', '19 Apr 2026', '1930 UTC')

[2] SANGAY ECUADOR
    Código: SANGAY3
    Fecha:  19 Apr 2026
    Hora:   1859 UTC
    Copiar: generador.procesar_volcan('SANGAY ECUADOR', 'SANGAY3', '19 Apr 2026', '1859 UTC')

[3] FUEGO GUATEMALA
    Código: FUEG
    Fecha:  19 Apr 2026
    Hora:   1714 UTC
    Copiar: generador.procesar_volcan('FUEGO GUATEMALA', 'FUEG', '19 Apr 2026', '1714 UTC')

[4] REVENTADOR ECUADOR
    Código: REVE
    Fecha:  19 Apr 2026
    Hora:   1410 UTC
    Copiar: generador.procesar_volcan('REVENTADOR ECUADOR', 'REVE', '19 Apr 2026', '1410 UTC')

[5] SANTA MARIA GUATEMALA
    Código: SANT
    Fecha:  19 Apr 2026
    Hora:   1355 UTC
    Copiar: generador.procesar_volcan('SANTA MARIA GUATEMALA', '

In [37]:
# O procesar todos en lista:
generador = VAACDocumentGenerator()

# generador.procesar_volcan('SANTA MARIA GUATEMALA', 'SANT', '19 Apr 2026', '1930 UTC')


Procesando: SANTA MARIA GUATEMALA - 1930 UTC
URL generada: https://www.ospo.noaa.gov/VAAC/ARCH26/SANTAMA/2026D191930.html
  ✓ Imagen agregada correctamente
✓ Documento creado: D:/OVM/2-REGISTROS-VA_ADVISORY/2-OTROS\AÑO-2026\GUATEMALA\SANTA MARIA\4-ABR_2026\SANTA_ADV_0183.docx


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.22s/it]

✓ PDF generado: \\PC_GOA76\solicitud_carpeta\SANTA_ADV_0183.pdf


True